In [16]:
import pandas as pd
import numpy as np
import matplotlib as plt
import ast
from collections import Counter


import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
pd.set_option('display.max_columns',100)          
pd.set_option('display.max_rows',100) 
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error





In [30]:
df_inside = pd.read_csv("paris_complete_seasonal_dataset.csv", encoding='ISO-8859-1',low_memory=False)


0         0
1         0
2         1
3         0
4         0
         ..
109613    0
109614    0
109615    0
109616    0
109617    1
Name: superhost, Length: 109618, dtype: int64

In [ ]:
#df_inside = pd.read_csv("../data/insideAirBnBdataset1.csv", encoding='ISO-8859-1',low_memory=False)
###feature selection and pre processing
primary_columns_inside = df_inside.columns.copy()


df_inside = df_inside[primary_columns_inside].copy()

df_inside.dropna(subset=primary_columns_inside, inplace=True)

bool_cols = [
    "host_is_superhost", "host_has_profile_pic",
    "host_identity_verified", "instant_bookable"
]

for col in bool_cols:
    df_inside[col] = df_inside[col].map({True: 1, False: 0, 't':1, 'f':0}).astype(int)

df_inside['price'] = (
    df_inside['price']
    .replace('[\$,]', '', regex=True)
    .astype(float)
)
df_inside['num_amenities'] = df_inside['amenities'].apply(lambda x: len(eval(x)) if isinstance(x, str) else 0)

from scipy.stats import zscore
numeric_cols = df_inside.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('price')  # keep target separate

z_scores = np.abs(zscore(df_inside[numeric_cols]))
outliers = (z_scores > 3).any(axis=1)
print(f"Outliers detected: {outliers.sum()} rows")
for col in numeric_cols:
    lower = df_inside[col].quantile(0.01)
    upper = df_inside[col].quantile(0.99)
    df_inside[col] = df_inside[col].clip(lower, upper)



<>:28: SyntaxWarning: invalid escape sequence '\$'
<>:28: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_39731/3485152127.py:28: SyntaxWarning: invalid escape sequence '\$'
  .replace('[\$,]', '', regex=True)
/tmp/ipykernel_39731/3485152127.py:28: SyntaxWarning: invalid escape sequence '\$'
  .replace('[\$,]', '', regex=True)


KeyError: "['host_is_superhost', 'host_total_listings_count', 'host_has_profile_pic', 'host_identity_verified', 'accommodates', 'bedrooms', 'minimum_nights', 'maximum_nights', 'amenities', 'review_scores_cleanliness', 'review_scores_checkin', 'last_scraped', 'review_scores_communication', 'review_scores_location', 'review_scores_value'] not in index"

In [18]:
df_inside['temporal_date'] = pd.to_datetime(df_inside['last_scraped'])

def assign_quarter(month):
    if month in [1,2,3]:
        return 'Q1_Mar'
    elif month in [4,5,6]:
        return 'Q2_Jun'
    elif month in [7,8,9]:
        return 'Q3_Sep'
    else:
        return 'Q4_Dec'

df_inside['quarter'] = df_inside['temporal_date'].dt.month.apply(assign_quarter)

df_inside['quarter_cat'] = df_inside['quarter'].astype('category').cat.codes
df_inside = df_inside.drop(columns=["temporal_date",'quarter','amenities'])

In [20]:
df_inside["last_scraped"].unique()

array(['2025-03-11', '2025-03-07', '2025-03-14', '2025-03-03',
       '2025-03-13', '2025-03-04', '2025-03-16', '2025-03-06',
       '2025-03-12', '2025-03-05', '2025-03-10', '2025-03-15',
       '2025-03-09', '2025-03-08'], dtype=object)

In [31]:
X = df_inside.drop(columns=["price"])
y = df_inside["price"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)



(87694, 156) (10962, 156) (10962, 156)


In [ ]:
#scaler = StandardScaler()
#X_train_scaled = scaler.fit_transform(X_train)
#X_val_scaled = scaler.transform(X_val)
#X_test_scaled = scaler.transform(X_test)

#pca = PCA()
#X_train_pca = pca.fit_transform(X_train_scaled)


In [32]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns

num_cols, cat_cols


(Index(['id', 'latitude', 'longitude', 'month', 'host_years', 'superhost',
        'instant_bookable', 'is_entire_home', 'number_of_reviews',
        'review_scores_rating',
        ...
        'has_sun_loungers', 'has_tv', 'has_tv_with_standard_cable',
        'has_toaster', 'has_trash_compactor', 'has_washer',
        'has_washer_âÂ in_unit', 'has_wifi', 'has_window_guards',
        'has_wine_glasses'],
       dtype='object', length=156),
 Index([], dtype='object'))

In [33]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

def evaluate(model, X, y, name):
    preds = model.predict(X)
    print(name)
    print("MAE:", mean_absolute_error(y, preds))
    print("RMSE:", root_mean_squared_error(y, preds))
    print("R2:", r2_score(y, preds))
    print("-" * 30)




In [34]:
ols = Pipeline([
    ("prep", preprocessor),
    ("model", LinearRegression())
])

ols.fit(X_train, y_train)
evaluate(ols, X_val, y_val, "OLS")


OLS
MAE: 192.2024084897931
RMSE: 567.5896758251353
R2: 0.07325143552064717
------------------------------


In [35]:
ridge = Pipeline([
    ("prep", preprocessor),
    ("model", Ridge(alpha=1.0))
])

lasso = Pipeline([
    ("prep", preprocessor),
    ("model", Lasso(alpha=0.01))
])

ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

evaluate(ridge, X_val, y_val, "Ridge")
evaluate(lasso, X_val, y_val, "Lasso")


Ridge
MAE: 192.20171989161256
RMSE: 567.5896475052692
R2: 0.07325152800081591
------------------------------
Lasso
MAE: 192.16677049108162
RMSE: 567.587734384572
R2: 0.07325777539698597
------------------------------


In [36]:
rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
evaluate(rf, X_val, y_val, "Random Forest")


Random Forest
MAE: 112.53715447998832
RMSE: 431.8266629651774
R2: 0.4635714847967555
------------------------------


In [37]:
gbr = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

gbr.fit(X_train, y_train)
evaluate(gbr, X_val, y_val, "Gradient Boosting")


Gradient Boosting
MAE: 160.7590445763615
RMSE: 524.3250832528431
R2: 0.2091498225041839
------------------------------


In [38]:
mlp = Pipeline([
    ("prep", preprocessor),
    ("model", MLPRegressor(
        hidden_layer_sizes=(128, 64),
        max_iter=300,
        random_state=42
    ))
])

mlp.fit(X_train, y_train)
evaluate(mlp, X_val, y_val, "Neural Network")


Neural Network
MAE: 194.539533556248
RMSE: 508.18240876985544
R2: 0.25709684563079627
------------------------------


/home/alant/.local/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

def accuracy_within_ranges(y_true, y_pred, euro_ranges, pct_ranges):
    abs_err = np.abs(y_true - y_pred)
    pct_err = abs_err / y_true

    abs_acc = {
        f"±€{r}": np.mean(abs_err <= r) * 100
        for r in euro_ranges
    }
    pct_acc = {
        f"±{int(p*100)}%": np.mean(pct_err <= p) * 100
        for p in pct_ranges
    }
    return abs_acc, pct_acc


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
    oob_score=True
)

param_grid = {
    "n_estimators": [200],
    "max_depth": [15, 20, None],
    "min_samples_split": [5, 10],
    "min_samples_leaf": [3, 5]
}

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
grid.best_params_


In [ ]:
from sklearn.model_selection import train_test_split

X_trainval = np.vstack([X_train, X_val])
y_trainval = np.hstack([y_train, y_val])

best_rf.fit(X_trainval, y_trainval)


In [ ]:
y_pred_test = best_rf.predict(X_test)

metrics_test = regression_metrics(y_test, y_pred_test)
metrics_test


In [ ]:
best_rf.oob_score_


In [ ]:
euro_ranges = [100, 200, 500, 1000, 2000]
pct_ranges = [0.10, 0.20, 0.30, 0.50]

abs_acc, pct_acc = accuracy_within_ranges(
    y_test, y_pred_test,
    euro_ranges, pct_ranges
)

abs_acc, pct_acc


In [ ]:
import pandas as pd

importances = pd.Series(
    best_rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importances.head(15)
